# Hackathon séries temporelles : notebook de démarrage

**Tâche.** On dispose de mesures horaires de 14 variables météo d'une station (températures, pression, humidités, pression de vapeur, densité de l'air, vent). Pour chaque bloc du test, on connaît les **72 dernières heures** des 14 variables et on doit prévoir la **température de l'air `temp_1_C` pour les 24 heures suivantes**.

**Métrique.** RMSE (en °C) sur les 24 heures de tous les blocs.

**Fichiers.**
- `train.csv` : série horaire continue (`t` = indice d'heure), avec `heure`, `jour_semaine`, `jour_annee` et les 14 variables.
- `test.csv` : un bloc par `bloc_id`, 72 lignes chacun (`pas` de -72 à -1).
- `sample_submission.csv` : une ligne par bloc et par horizon, `id = <bloc_id>_<horizon 01..24>`.

**Règles.** Données externes interdites. Votre notebook doit reproduire exactement le fichier soumis.

Ce notebook fournit : le chargement, une validation qui imite le test, deux baselines naïves, **SARIMA** (soumis par défaut), le découpage des données en exemples et l'écriture de la soumission. **À vous** : la normalisation et votre modèle (reservoir / ESN, GRU, LSTM…).

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# Sur Kaggle, les données sont quelque part sous /kaggle/input/ : on cherche train.csv
_found = sorted(Path('/kaggle/input').rglob('train.csv')) if Path('/kaggle/input').exists() else []
DATA = _found[0].parent if _found else Path('.')
print('Données lues dans :', DATA.resolve())
train = pd.read_csv(DATA / 'train.csv')
test = pd.read_csv(DATA / 'test.csv')
sample = pd.read_csv(DATA / 'sample_submission.csv')
TARGET, N_IN, N_OUT = 'temp_1_C', 72, 24
VARS = [c for c in train.columns if c not in ('t', 'heure', 'jour_semaine', 'jour_annee')]
print(train.shape, test.shape, sample.shape)
train.head()

In [ ]:
# Exploration rapide : la cible sur 2 semaines et sur toute la période
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 1, figsize=(12, 6))
train[TARGET].iloc[:24 * 14].plot(ax=ax[0], title='2 semaines')
train[TARGET].rolling(24 * 7).mean().plot(ax=ax[1], title='moyenne glissante 7 jours')
plt.tight_layout()

## Validation qui imite le test

On découpe la **dernière année** du train en blocs de 96 h (72 h d'entrée, 24 h à prévoir), comme le test. On entraîne sur ce qui précède. Ne mélangez jamais les dates : une validation aléatoire surestime fortement vos scores.

In [ ]:
VAL_START = len(train) - 24 * 365
fit_df = train.iloc[:VAL_START]
val_df = train.iloc[VAL_START:]

def make_blocks(df):
    blocks, targets = [], []
    for s in range(0, len(df) - N_IN - N_OUT + 1, N_IN + N_OUT):
        blocks.append(df.iloc[s:s + N_IN])
        targets.append(df[TARGET].to_numpy()[s + N_IN:s + N_IN + N_OUT])
    return blocks, np.stack(targets)

val_blocks, val_y = make_blocks(val_df)
rmse = lambda a, b: float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))
print(len(val_blocks), 'blocs de validation')

## Baselines naïves

Une fonction de prévision prend un bloc de 72 h (DataFrame) et renvoie 24 valeurs.

In [ ]:
def predict_persistance(block):
    return np.repeat(block[TARGET].iloc[-1], N_OUT)

def predict_naif_saisonnier(block):
    return block[TARGET].to_numpy()[-24:]

for f in (predict_persistance, predict_naif_saisonnier):
    pred = np.stack([f(b) for b in val_blocks])
    print(f.__name__, 'RMSE validation =', round(rmse(pred, val_y), 3))

## Baseline SARIMA (soumise par défaut)

SARIMA(1,0,1)(1,1,0) avec une saisonnalité de 24 h, **univarié** (température seule), estimé sur les 120 derniers jours. Il est ensuite appliqué à chaque bloc de 72 h avec `res.apply(...)`, sans réestimation.

C'est la référence classique à battre : soumettre ce modèle tel quel donne environ **8/20**. Pour viser une meilleure note, il faut faire mieux que SARIMA avec votre propre modèle (ESN, GRU, LSTM…).

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

def fit_sarima(series):
    y = series.to_numpy()[-24 * 120:]                  # 120 derniers jours
    return SARIMAX(y, order=(1, 0, 1), seasonal_order=(1, 1, 0, 24)).fit(disp=False)

# Validation : estimé sur la partie d'entraînement seulement
sarima_val = fit_sarima(fit_df[TARGET])
pred = np.stack([sarima_val.apply(b[TARGET].to_numpy()).forecast(N_OUT) for b in val_blocks])
print('SARIMA RMSE validation =', round(rmse(pred, val_y), 3))

# Soumission : réestimé sur la fin du train complet
sarima_full = fit_sarima(train[TARGET])
def predict_sarima(block):
    return sarima_full.apply(block[TARGET].to_numpy()).forecast(N_OUT)

## Préparer les exemples pour votre modèle

`make_windows` découpe une série continue en exemples, comme dans le test : `X` contient 72 h des variables choisies, `Y` les 24 températures suivantes. Les valeurs sont **brutes** (non normalisées) : la normalisation est à faire par vous, avec des statistiques calculées sur la partie d'entraînement seulement.

In [ ]:
FEATURES = VARS                     # à adapter : variables d'entrée (vous pouvez en ajouter, par ex. l'heure codée en sinus/cosinus)

def make_windows(df, stride=1):
    """Série continue -> X (exemples, 72, variables), Y (exemples, 24)."""
    v = df[FEATURES].to_numpy(np.float32)
    t = df[TARGET].to_numpy(np.float32)
    idx = range(N_IN, len(df) - N_OUT + 1, stride)
    return np.stack([v[i - N_IN:i] for i in idx]), np.stack([t[i:i + N_OUT] for i in idx])

X_fit, Y_fit = make_windows(fit_df)                                      # entraînement
X_val = np.stack([b[FEATURES].to_numpy(np.float32) for b in val_blocks])  # validation : mêmes blocs que plus haut
print('X_fit', X_fit.shape, ' Y_fit', Y_fit.shape, ' X_val', X_val.shape, ' val_y', val_y.shape)

## Votre modèle

Écrivez la fonction `predict_mon_modele` : elle reçoit un bloc de 72 h (DataFrame, mêmes colonnes que `test.csv`) et renvoie les 24 températures prévues. Vous pouvez reprendre et adapter le code de vos cours (ESN, GRU, LSTM). Évaluez-la avec la validation, puis soumettez-la (dernière cellule).

In [ ]:
def predict_mon_modele(block):
    # block : DataFrame de 72 lignes ; renvoyer un tableau de 24 valeurs de temp_1_C
    raise NotImplementedError('À écrire : votre modèle')

# Validation (décommentez quand votre modèle est prêt) :
# pred = np.stack([predict_mon_modele(b) for b in val_blocks])
# print('Mon modèle  RMSE validation =', round(rmse(pred, val_y), 3))

## Pistes

1. **Décomposition / SARIMA** : autres ordres, décomposition `STL`, variables exogènes (`SARIMAX(..., exog=...)`).
2. **Normalisation** des variables, encodage cyclique de `heure` et `jour_annee`, direction du vent en sinus/cosinus.
3. **Reservoir computing (ESN)**, **GRU**, **LSTM** : prévision directe des 24 h, ou prévision heure par heure répétée 24 fois.
4. **Hybride** : retirer la saisonnalité (ou la prévision SARIMA) et modéliser le résidu avec un RNN.

Comparez toutes les méthodes avec la **même** validation, et notez les temps d'entraînement.

## Soumission

In [ ]:
PREDICT = predict_sarima   # par défaut SARIMA (≈ 8/20) ; remplacez par predict_mon_modele

rows = []
for bid, block in test.groupby('bloc_id', sort=False):
    pred = PREDICT(block)
    rows += [{'id': f'{bid}_{k + 1:02d}', TARGET: float(pred[k])} for k in range(N_OUT)]
sub = pd.DataFrame(rows)
assert len(sub) == len(sample) and set(sub.id) == set(sample.id)
sub.to_csv('submission.csv', index=False)
sub.head()